<a href="https://colab.research.google.com/github/estevamjr/ticket-andon-it/blob/main/%5BCyberBison%5D_Ticket_Andon_IT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚨 Project: Ticket-Andon-IT - Machine Learning Pipeline
This notebook demonstrates the development of an intelligent monitoring system for IT assets. Using the **Andon** philosophy from Lean Manufacturing, we classify real-time telemetry logs into three categories: **Normal (0)**, **Warning (1)**, and **Critical (2)**.

In [ ]:
import pandas as pd
import requests
import joblib
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Algorithms
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# 1. Data Ingestion from GitHub (JSON)
url = "https://raw.githubusercontent.com/estevamjr/ticket-andon-it/main/data_logs.json"
response = requests.get(url)
data = response.json()

# Flattening the nested JSON structure
df = pd.json_normalize(data)

# Feature Selection
X = df[['metrics.cpu_usage_pct', 'metrics.mem_available_gb', 'metrics.active_threats', 'metrics.untrusted_processes']]
y = df['status_andon']

print(f"Dataset loaded. Total records: {len(df)}")
X.head()

Dataset loaded. Total records: 1500


,metrics.cpu_usage_pct,metrics.mem_available_gb,metrics.active_threats,metrics.untrusted_processes
0,21.55,10.28,0,1
1,33.02,11.59,0,1
2,37.67,6.54,0,0
3,54.64,1.68,3,9
4,34.63,6.10,0,0


## 2. Data Pre-processing and Holdout
We apply a **Holdout** strategy, splitting the data into **Training (80%)** and **Testing (20%)** sets to evaluate the model on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Data split completed.")

Data split completed.


## 3. Competitive Modeling (Model Selection)
We evaluate four classic algorithms using **StandardScaler** and **10-fold Cross-Validation** to find the most robust model.

In [ ]:
models = [
    ('KNN', KNeighborsClassifier()),
    ('CART', DecisionTreeClassifier()),
    ('NB', GaussianNB()),
    ('SVM', SVC())
]

for name, model in models:
    pipe = Pipeline([('scaler', StandardScaler()), ('classifier', model)])
    cv_results = cross_val_score(pipe, X_train, y_train, cv=10, scoring='accuracy')
    print(f"{name}: Mean Accuracy = {cv_results.mean():.4f}")

KNN: Mean Accuracy = 1.0000
CART: Mean Accuracy = 1.0000
NB: Mean Accuracy = 1.0000
SVM: Mean Accuracy = 1.0000


## 4. Hyperparameter Optimization
We perform a **Grid Search** to fine-tune the SVM model parameters for better precision.

In [ ]:
pipe_svm = Pipeline([('scaler', StandardScaler()), ('svm', SVC())])
param_grid = {'svm__C': [0.1, 1, 10], 'svm__kernel': ['rbf', 'linear']}

grid_search = GridSearchCV(pipe_svm, param_grid, cv=5)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best Hyperparameters:", grid_search.best_params_)

Best Hyperparameters: {'svm__C': 0.1, 'svm__kernel': 'linear'}


## 5. Final Evaluation and Export
Final test set prediction and exporting the model as a `.pkl` file for the backend.

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

# Exporting the model
joblib.dump(best_model, 'modelo_andon.pkl')
print("Model saved as 'modelo_andon.pkl'")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       211
           1       1.00      1.00      1.00        60
           2       1.00      1.00      1.00        29

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

Model saved as 'modelo_andon.pkl'


### 6. Reflexão sobre Segurança e LGPD (Desenvolvimento Seguro)
Em conformidade com as boas práticas de Engenharia de Software e a LGPD, o dataset utilizado neste projeto passou por um processo de **anonimização**. Os nomes das máquinas físicas, IPs reais e credenciais de usuários foram removidos ou ofuscados (ex: substituídos por "SENS-01", "WS-02"). A telemetria coleta estritamente dados de performance de hardware e processos, garantindo que nenhuma informação pessoal ou sensível dos colaboradores trafegue pelo pipeline de Machine Learning.

### 7. Conclusão e Análise de Resultados
A etapa de modelagem competitiva demonstrou que os algoritmos clássicos performaram excepcionalmente bem, alcançando acurácia máxima. Isso se deve à linearidade e aos padrões bem definidos do dataset de telemetria de TI. Optamos pelo **SVM** (com Kernel Linear) após a otimização de hiperparâmetros via GridSearch, pois ele apresenta grande robustez em cenários de classificação de métricas de infraestrutura.
* **Ponto de Atenção:** Como o modelo alcançou 100% no teste, existe o risco de *Overfitting* se aplicado a uma infraestrutura com características muito diferentes da atual. Para um ambiente de produção real (deploy), recomenda-se o re-treinamento periódico do `.pkl` com novos lotes de logs para garantir a generalização do modelo ao longo do tempo.